# Assignment Beta, Question 3: Knowledge Distillation
**Student ID:** 23k-0078

Prompt: "The patient was admitted to the ___". Vocabulary: hospital, clinic, ward, school.

Loss: $L = \alpha T^2\, KL(p_T \| q_T) + (1-\alpha)\, CE(y, q_1)$, with $T=2$, $\alpha=0.5$ and natural logarithms.

In [1]:
import numpy as np
np.set_printoptions(precision=6, suppress=True)

words = ["hospital", "clinic", "ward", "school"]
z_t = np.array([5.0, 3.0, 2.0, 0.0])
z_s = np.array([3.0, 2.5, 1.0, 0.5])
y = np.array([1.0, 0.0, 0.0, 0.0])
T, alpha, eta = 2.0, 0.5, 0.5

def softmax(z, temperature=1.0):
    e = np.exp(z / temperature)
    return e / e.sum()

def loss_parts(student_logits):
    p_T = softmax(z_t, T)
    q_T = softmax(student_logits, T)
    q_1 = softmax(student_logits)
    kl = np.sum(p_T * np.log(p_T / q_T))
    ce = -np.sum(y * np.log(q_1))
    return kl, ce, alpha * T**2 * kl + (1 - alpha) * ce

## (a) Probabilities at T = 1
$p_i = e^{z_i} / \sum_j e^{z_j}$

In [2]:
p_1, q_1 = softmax(z_t), softmax(z_s)
print("Teacher exponentials:", np.exp(z_t), "sum =", np.exp(z_t).sum().round(6))
print("Student exponentials:", np.exp(z_s), "sum =", np.exp(z_s).sum().round(6))
print("Teacher p1:", p_1)
print("Student q1:", q_1)

Teacher exponentials: [148.413159  20.085537   7.389056   1.      ] sum = 176.887752
Student exponentials: [20.085537 12.182494  2.718282  1.648721] sum = 36.635034
Teacher p1: [0.839025 0.11355  0.041773 0.005653]
Student q1: [0.54826  0.332537 0.074199 0.045004]


## (b) Softened probabilities at T = 2
$p_i = e^{z_i/T} / \sum_j e^{z_j/T}$

In [3]:
p_T, q_T = softmax(z_t, T), softmax(z_s, T)
print("Teacher z/T:", z_t / T, "exp:", np.exp(z_t / T), "sum =", np.exp(z_t / T).sum().round(6))
print("Student z/T:", z_s / T, "exp:", np.exp(z_s / T), "sum =", np.exp(z_s / T).sum().round(6))
print("Teacher pT:", p_T)
print("Student qT:", q_T)

Teacher z/T: [2.5 1.5 1.  0. ] exp: [12.182494  4.481689  2.718282  1.      ] sum = 20.382465
Student z/T: [1.5  1.25 0.5  0.25] exp: [4.481689 3.490343 1.648721 1.284025] sum = 10.904779
Teacher pT: [0.597695 0.21988  0.133364 0.049062]
Student qT: [0.410984 0.320075 0.151193 0.117749]


## (c) KL divergence
$KL(p_T \| q_T) = \sum_i p_{T,i} \ln(p_{T,i} / q_{T,i})$

In [4]:
terms = p_T * np.log(p_T / q_T)
for w, a, b, t in zip(words, p_T, q_T, terms):
    print(f"{w:9s} {a:.6f} * ln({a:.6f} / {b:.6f}) = {t:+.6f}")
kl = terms.sum()
print("KL =", round(kl, 6))

hospital  0.597695 * ln(0.597695 / 0.410984) = +0.223852
clinic    0.219880 * ln(0.219880 / 0.320075) = -0.082559
ward      0.133364 * ln(0.133364 / 0.151193) = -0.016734
school    0.049062 * ln(0.049062 / 0.117749) = -0.042952
KL = 0.081607


## (d) Hard-label cross-entropy
Only the correct class contributes: $CE = -\ln q_{1,\text{hospital}}$

In [5]:
ce = -np.log(q_1[0])
print(f"CE = -ln({q_1[0]:.6f}) = {ce:.6f}")

CE = -ln(0.548260) = 0.601005


## (e) Total distillation loss

In [6]:
L = alpha * T**2 * kl + (1 - alpha) * ce
print(f"L = {alpha} * {T**2} * {kl:.6f} + {1 - alpha} * {ce:.6f}")
print(f"  = {alpha * T**2 * kl:.6f} + {(1 - alpha) * ce:.6f} = {L:.6f}")

L = 0.5 * 4.0 * 0.081607 + 0.5 * 0.601005
  = 0.163215 + 0.300502 = 0.463717


## (f) Gradient with respect to the student's logits
The teacher's probabilities are constant, so $\partial KL / \partial z_s = \frac{1}{T}(q_T - p_T)$, and $\partial CE / \partial z_s = q_1 - y$.

$$\nabla_{z_s} L = \alpha T^2 \cdot \frac{1}{T}(q_T - p_T) + (1-\alpha)(q_1 - y) = \alpha T (q_T - p_T) + (1-\alpha)(q_1 - y)$$

With $\alpha T = 1$ and $1-\alpha = 0.5$, this becomes $(q_T - p_T) + 0.5\,(q_1 - y)$.

In [7]:
distill_term = alpha * T * (q_T - p_T)
hard_term = (1 - alpha) * (q_1 - y)
grad = distill_term + hard_term
print("alpha*T*(qT - pT):", distill_term)
print("(1-alpha)*(q1 - y):", hard_term)
print("Gradient:", grad)

# Independent check by central finite differences.
h = 1e-6
numeric = np.array([(loss_parts(z_s + h * e)[2] - loss_parts(z_s - h * e)[2]) / (2 * h) for e in np.eye(4)])
print("Numerical gradient:", numeric)
print("Largest difference:", np.abs(grad - numeric).max())
print("Gradient components sum to:", round(grad.sum(), 10))

alpha*T*(qT - pT): [-0.186711  0.100195  0.017829  0.068687]
(1-alpha)*(q1 - y): [-0.22587   0.166268  0.037099  0.022502]
Gradient: [-0.412581  0.266463  0.054928  0.091189]
Numerical gradient: [-0.412581  0.266463  0.054928  0.091189]
Largest difference: 1.9252188732110653e-10
Gradient components sum to: -0.0


## (g) One gradient-descent step, $\eta = 0.5$
$z_s' = z_s - \eta \nabla_{z_s} L$

In [8]:
z_new = z_s - eta * grad
kl_new, ce_new, L_new = loss_parts(z_new)
print("New student logits:", z_new)
print("New qT:", softmax(z_new, T))
print("New q1:", softmax(z_new))
print(f"New KL = {kl_new:.6f}, new CE = {ce_new:.6f}")
print(f"New L = {alpha * T**2 * kl_new:.6f} + {(1 - alpha) * ce_new:.6f} = {L_new:.6f}")
print(f"Loss change: {L:.6f} -> {L_new:.6f} ({L_new - L:+.6f}, {(L_new - L) / L * 100:+.2f}%)")

New student logits: [3.20629  2.366768 0.972536 0.454405]
New qT: [0.447006 0.293774 0.146305 0.112914]
New q1: [0.623889 0.269468 0.066834 0.039809]
New KL = 0.056682, new CE = 0.471783
New L = 0.113364 + 0.235892 = 0.349256
Loss change: 0.463717 -> 0.349256 (-0.114461, -24.68%)


**Interpretation.** Only the gradient for *hospital* is negative, so one step raises that logit and lowers the other three. The student's probability for the correct word rises from 0.5483 to 0.6239, and both loss terms fall.